# Chiffrer et déchiffrer

| Module | Durée | Prérequis |
|---|---|---|
| M4 – Inverse, déterminant, rang | 75 min | M2 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Lou et Malo veulent échanger des messages que leurs camarades ne pourront pas lire. Ils découvrent le **chiffre de Hill**, inventé par le mathématicien Lester Hill en 1929 : on remplace les lettres par des nombres, on les groupe par deux, et on multiplie chaque paire par une matrice secrète, la **clé**.

Malo choisit sa clé au hasard. Lou réussit à chiffrer un message avec, mais **personne ne peut le déchiffrer**, pas même Malo.

**Les questions : comment chiffrer et déchiffrer avec une matrice ? Pourquoi certaines clés sont-elles inutilisables, et comment les reconnaître ? Ce chiffre est-il sûr ?**

**Hypothèses.** On n'utilise que les 26 lettres majuscules : on retire les accents, les espaces et la ponctuation. Si le message a un nombre impair de lettres, on ajoute un X. Ces conventions posent-elles problème ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

Parmi toutes les matrices $2 \times 2$ à coefficients entre 0 et 25, quelle proportion peut servir de clé ?

In [ ]:
pari_proportion_cles_valides = 0.9    # entre 0 et 1 (modifiez)

## ③ Abstraire

### Chiffrer

On numérote les lettres : A = 0, B = 1, …, Z = 25. Chaque paire de lettres devient un vecteur $\mathbf{p}$, chiffré en

$$
\mathbf{c} = K\,\mathbf{p} \pmod{26}
$$

« Modulo 26 » signifie qu'on garde le reste de la division par 26, pour retomber sur une lettre.

![Schéma : le message devient des nombres, groupés par paires, multipliés par K modulo 26 ; pour déchiffrer, on multiplie par l'inverse de K modulo 26](../../assets/schemas/schema-hill.svg)

### Déchiffrer : il faut un inverse

Pour retrouver $\mathbf{p}$, il faut une matrice $K^{-1}$ telle que $K^{-1} K = I$ **modulo 26**. La formule du notebook 4.1 s'adapte :

$$
K^{-1} = (\det K)^{-1} \begin{pmatrix} d & -b \\ -c & a \end{pmatrix} \pmod{26}
$$

où $(\det K)^{-1}$ est l'**inverse modulo 26** du déterminant : le nombre $u$ tel que $u \times \det K \equiv 1 \pmod{26}$.

**Le concept s'applique-t-il ?** Dans les nombres réels, une matrice est inversible dès que $\det K \neq 0$. Mais ici, on calcule modulo 26, et la règle change : un nombre n'a d'inverse modulo 26 que s'il n'a aucun diviseur commun avec $26 = 2 \times 13$.

$$
K \text{ est une clé valide} \iff \operatorname{pgcd}(\det K,\ 26) = 1
$$

**Activité** : Avec $K = \begin{pmatrix} 3 & 3 \\ 2 & 5 \end{pmatrix}$, calculez $\det K$. Vérifiez que $3 \times 9 = 27 \equiv 1 \pmod{26}$, puis écrivez $K^{-1}$ modulo 26. Chiffrez la paire « HI ».

Indiquez vos réponses et justifiez-les.

## ④ Calculer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import unicodedata
from math import gcd

def nettoyer(texte):
    # Majuscules sans accents, lettres seulement, longueur paire
    texte = unicodedata.normalize("NFD", texte.upper())
    lettres = [c for c in texte if "A" <= c <= "Z"]
    if len(lettres) % 2:
        lettres.append("X")
    return "".join(lettres)

def vers_nombres(texte):
    return np.array([ord(c) - ord("A") for c in texte])

def vers_texte(nombres):
    return "".join(chr(int(n) + ord("A")) for n in nombres)

def chiffrer(texte, K):
    p = vers_nombres(nettoyer(texte)).reshape(-1, 2).T    # une paire par colonne : forme (2, n/2)
    c = (K @ p) % 26
    return vers_texte(c.T.flatten())

def inverse_mod26(K):
    (a, b), (c, d) = K
    det = int(a * d - b * c) % 26
    if gcd(det, 26) != 1:
        raise ValueError(f"det K = {det} (mod 26) a un diviseur commun avec 26 : la clé n'est pas inversible.")
    inv_det = pow(det, -1, 26)                            # inverse modulaire (Python 3.8 et plus)
    return (inv_det * np.array([[d, -b], [-c, a]])) % 26

def dechiffrer(texte_chiffre, K):
    return chiffrer(texte_chiffre, inverse_mod26(K))

K = np.array([[3, 3], [2, 5]])
message = "Rendez-vous samedi à la bibliothèque !"
secret = chiffrer(message, K)
print("Message nettoyé :", nettoyer(message))
print("Message chiffré :", secret)
print("K^-1 mod 26     :", inverse_mod26(K).tolist())
print("Déchiffré       :", dechiffrer(secret, K))

### La clé de Malo

In [ ]:
K_malo = np.array([[2, 4], [1, 3]])
print("det réel de la clé de Malo :", round(np.linalg.det(K_malo)), " -> inversible dans les réels")
print("Chiffrage de « AANN » :", chiffrer("AANN", K_malo))
print("Chiffrage de « AAAA » :", chiffrer("AAAA", K_malo))
try:
    dechiffrer(chiffrer("BONJOUR", K_malo), K_malo)
except ValueError as erreur:
    print("Déchiffrement impossible :", erreur)

**Activité** : Deux messages différents donnent le même texte chiffré. Pourquoi est-ce exactement la raison pour laquelle on ne peut pas déchiffrer ?

Indiquez vos réponses et justifiez-les.

### Combien de clés sont valides ?

Il y a $26^4 = 456\,976$ matrices à coefficients entre 0 et 25. Testons-les toutes, en une seule opération vectorisée.

In [ ]:
a, b, c, d = np.meshgrid(*[np.arange(26)] * 4, indexing="ij")
det = (a * d - b * c) % 26
valides = (det % 2 != 0) & (det % 13 != 0)
print(f"Clés valides : {valides.sum()} sur {valides.size}, soit {valides.mean():.1%}")

fig, ax = plt.subplots(figsize=(8, 3.5))
compte = np.bincount(det.ravel(), minlength=26)
couleurs = [LIOUAI["vert"] if gcd(k, 26) == 1 else LIOUAI["gris"] for k in range(26)]
ax.bar(range(26), compte, color=couleurs)
ax.set_xlabel("det K modulo 26"); ax.set_ylabel("nombre de matrices")
ax.set_title("Valeurs du déterminant : en vert, celles qui donnent une clé valide")
plt.tight_layout(); plt.show()

### Une faiblesse : l'attaque à texte clair connu

Les messages de Lou commencent toujours par « RENDEZ-VOUS ». Une camarade curieuse intercepte un message chiffré. Elle connaît donc deux paires en clair, « RE » et « ND », et les deux paires chiffrées correspondantes. Le chiffre étant linéaire, cela suffit :

$$
K\,P = C \pmod{26} \quad\Longrightarrow\quad K = C\,P^{-1} \pmod{26}
$$

où $P$ et $C$ ont pour colonnes les paires en clair et chiffrées.

In [ ]:
P_connu = vers_nombres("REND").reshape(-1, 2).T
C_intercepte = vers_nombres(secret[:4]).reshape(-1, 2).T
K_retrouvee = (C_intercepte @ inverse_mod26(P_connu)) % 26
print("Clé retrouvée :", K_retrouvee.tolist(), "  vraie clé :", K.tolist())
print("Tout le message :", dechiffrer(secret, K_retrouvee))

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_proportion_cles_valides:.0%} ; proportion réelle : {valides.mean():.1%}")

**Activité** :

1. Expliquez à Malo, sans formule, pourquoi sa clé ne marche pas, et donnez-lui une règle simple pour en choisir une bonne.
2. La clé de Malo est inversible dans les réels, mais pas modulo 26. Qu'est-ce qui a changé entre les deux situations ?
3. Pourquoi les déterminants pairs et le déterminant 13 sont-ils interdits, et pas les autres ?
4. Après l'attaque à texte clair connu, que conseiller à Lou et Malo ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Les limites du concept.** La linéarité rend le chiffre de Hill facile à utiliser, mais aussi facile à casser. Quelle autre faiblesse voyez-vous ? Pensez à une lettre répétée, ou à un message formé de paires identiques comme « AAAA ».
- **Hypothèses.** Le retrait des accents et l'ajout d'un X final changent-ils le sens d'un message ? Proposez un alphabet de 29 symboles (lettres, espace, point, virgule). Pourquoi 29 est-il un choix bien plus commode que 26 ?
- **Généraliser.** Écrivez `inverse_mod(K, m)` pour une matrice carrée de **n'importe quelle taille** et un module `m` quelconque. Indication : la formule générale est $K^{-1} = (\det K)^{-1}\operatorname{adj}(K)$, et `np.linalg.inv(K) * det` donne la matrice adjointe (à arrondir).

In [ ]:
def inverse_mod(K, m=26):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def inverse_mod(K, m=26):
    K = np.asarray(K)
    det = int(round(np.linalg.det(K)))
    if gcd(det % m, m) != 1:
        raise ValueError(f"det K = {det % m} (mod {m}) n'est pas inversible modulo {m}.")
    adjointe = np.round(np.linalg.inv(K) * det).astype(int)
    return (pow(det % m, -1, m) * adjointe) % m

K3 = np.array([[6, 24, 1], [13, 16, 10], [20, 17, 15]])      # clé 3 x 3 classique
print("K3^-1 mod 26 :", inverse_mod(K3).tolist())
print("Vérification :", ((K3 @ inverse_mod(K3)) % 26).tolist())
print("Avec 29 symboles (29 est premier), toute matrice de déterminant non multiple de 29 convient.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Reconnaître si le concept s'applique": 1,
    "Lister les sources d'erreur liées aux limites des concepts": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Combiner des outils pour produire le résultat": 1,
    "Énoncer le résultat dans le contexte du problème": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "4.2-chiffre-de-hill", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")